# Health Recommendation System

This notebook implements the health recommendation component of the
AI-Enhanced Disease Prediction and Health Recommendation System.

The recommendation system retrieves disease-specific health information,
including descriptions, precautions, medications, diet recommendations,
and workout guidance.

The retrieved information will later be provided as grounded context to
a Generative AI model to generate structured and user-friendly health guidance.

In [1]:
from pathlib import Path

import ast
import joblib
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
MODEL_DIR = PROJECT_ROOT / "models"

print("Project Root :", PROJECT_ROOT)
print("Data Directory:", DATA_DIR)
print("Model Directory:", MODEL_DIR)

Project Root : c:\Users\mohit\OneDrive\Desktop\AI HealthCare
Data Directory: c:\Users\mohit\OneDrive\Desktop\AI HealthCare\data\raw
Model Directory: c:\Users\mohit\OneDrive\Desktop\AI HealthCare\models


In [3]:
description_df = pd.read_csv(DATA_DIR / "description.csv")
precautions_df = pd.read_csv(DATA_DIR / "precautions.csv")
medications_df = pd.read_csv(DATA_DIR / "medications.csv")
diets_df = pd.read_csv(DATA_DIR / "diets.csv")
workout_df = pd.read_csv(DATA_DIR / "workout.csv")

print("Recommendation datasets loaded successfully!\n")

print("Description :", description_df.shape)
print("Precautions :", precautions_df.shape)
print("Medications :", medications_df.shape)
print("Diets       :", diets_df.shape)
print("Workout     :", workout_df.shape)

Recommendation datasets loaded successfully!

Description : (100, 2)
Precautions : (100, 5)
Medications : (100, 2)
Diets       : (100, 2)
Workout     : (100, 2)


In [4]:
recommendation_datasets = {
    "Description": description_df,
    "Precautions": precautions_df,
    "Medications": medications_df,
    "Diets": diets_df,
    "Workout": workout_df
}

for name, dataset in recommendation_datasets.items():
    print(f"\n{'=' * 60}")
    print(f"{name.upper()} DATASET")
    print(f"{'=' * 60}")

    print("Columns:")
    print(dataset.columns.tolist())

    print("\nFirst 3 rows:")
    display(dataset.head(3))


DESCRIPTION DATASET
Columns:
['Disease', 'Description']

First 3 rows:


,Disease,Description
0,Panic disorder,Panic disorder is a mental health condition ma...
1,Vaginitis,Vaginitis is inflammation of the vaginal tissu...
2,Problem during pregnancy,Problems during pregnancy refer to medical com...



PRECAUTIONS DATASET
Columns:
['Disease', 'Precaution_1', 'Precaution_2', 'Precaution_3', 'Precaution_4']

First 3 rows:


,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Panic disorder,Practice deep breathing,Avoid caffeine,Follow therapy plan,Seek support from loved ones
1,Vaginitis,Wear breathable cotton underwear,Avoid douching,Maintain genital hygiene,Avoid scented hygiene products
2,Problem during pregnancy,Attend regular prenatal visits,Avoid alcohol and smoking,Eat a balanced diet,Get adequate rest



MEDICATIONS DATASET
Columns:
['Disease', 'Medication']

First 3 rows:


,Disease,Medication
0,Panic disorder,"['SSRIs (e.g., Sertraline, Fluoxetine)', 'Benz..."
1,Vaginitis,"['Metronidazole', 'Clindamycin', 'Fluconazole'..."
2,Problem during pregnancy,"['Prenatal vitamins', 'Iron supplements', 'Ant..."



DIETS DATASET
Columns:
['Disease', 'Diet']

First 3 rows:


,Disease,Diet
0,Panic Disorder,"['Magnesium-rich foods (spinach, pumpkin seeds..."
1,Vaginitis,"['Probiotics (yogurt, kefir, sauerkraut)', 'Lo..."
2,Problem During Pregnancy,"['Prenatal vitamins (consult doctor)', 'Iron-r..."



WORKOUT DATASET
Columns:
['Disease', 'Workouts']

First 3 rows:


,Disease,Workouts
0,Panic disorder,"[""Deep breathing exercises: Calm your mind by ..."
1,Vaginitis,"[""Pelvic floor exercises: Strengthen pelvic mu..."
2,Problem during pregnancy,"[""Prenatal yoga: Gentle stretches safe for pre..."


In [5]:
def normalize_disease_name(name):
    return str(name).strip().lower()


for dataset in recommendation_datasets.values():
    dataset["Disease"] = dataset["Disease"].apply(normalize_disease_name)

print("Disease names normalized successfully.")

Disease names normalized successfully.


In [6]:
label_encoder = joblib.load(
    MODEL_DIR / "label_encoder.pkl"
)

model_diseases = {
    normalize_disease_name(disease)
    for disease in label_encoder.classes_
}

print("Diseases predicted by ML model:", len(model_diseases))
print()

for name, dataset in recommendation_datasets.items():

    dataset_diseases = set(dataset["Disease"])

    matched = model_diseases & dataset_diseases
    missing = model_diseases - dataset_diseases

    print(f"{name}:")
    print(f"  Available diseases : {len(dataset_diseases)}")
    print(f"  Matched diseases   : {len(matched)}")
    print(f"  Missing diseases   : {len(missing)}")

    if missing:
        print("  Missing:", sorted(missing))

    print()

Diseases predicted by ML model: 100

Description:
  Available diseases : 100
  Matched diseases   : 100
  Missing diseases   : 0

Precautions:
  Available diseases : 100
  Matched diseases   : 99
  Missing diseases   : 1
  Missing: ['chronic obstructive pulmonary disease (copd)']

Medications:
  Available diseases : 100
  Matched diseases   : 100
  Missing diseases   : 0

Diets:
  Available diseases : 100
  Matched diseases   : 100
  Missing diseases   : 0

Workout:
  Available diseases : 100
  Matched diseases   : 100
  Missing diseases   : 0



In [7]:
precaution_diseases = set(precautions_df["Disease"])

extra_precaution_diseases = precaution_diseases - model_diseases

print("Disease expected by model:")
print("chronic obstructive pulmonary disease (copd)")

print("\nUnmatched disease name in precautions dataset:")
print(sorted(extra_precaution_diseases))

Disease expected by model:
chronic obstructive pulmonary disease (copd)

Unmatched disease name in precautions dataset:
['copd']


In [8]:
disease_name_mapping = {
    "copd": "chronic obstructive pulmonary disease (copd)"
}

precautions_df["Disease"] = precautions_df["Disease"].replace(
    disease_name_mapping
)

print("Disease name mapping applied successfully.")

Disease name mapping applied successfully.


In [9]:
print("FINAL RECOMMENDATION DATA COVERAGE\n")

for name, dataset in recommendation_datasets.items():

    dataset_diseases = set(dataset["Disease"])

    matched = model_diseases & dataset_diseases
    missing = model_diseases - dataset_diseases

    print(
        f"{name:<12} : "
        f"{len(matched)}/{len(model_diseases)} diseases matched"
    )

    if missing:
        print("Missing:", sorted(missing))

FINAL RECOMMENDATION DATA COVERAGE

Description  : 100/100 diseases matched
Precautions  : 100/100 diseases matched
Medications  : 100/100 diseases matched
Diets        : 100/100 diseases matched
Workout      : 100/100 diseases matched


In [10]:
def parse_list(value):
    if isinstance(value, list):
        return value

    try:
        parsed_value = ast.literal_eval(value)

        if isinstance(parsed_value, list):
            return parsed_value

        return [parsed_value]

    except (ValueError, SyntaxError, TypeError):
        return [str(value)]


medications_df["Medication"] = medications_df["Medication"].apply(parse_list)
diets_df["Diet"] = diets_df["Diet"].apply(parse_list)
workout_df["Workouts"] = workout_df["Workouts"].apply(parse_list)

print("List-based recommendation columns parsed successfully.")

print("\nExample medication:")
print(medications_df["Medication"].iloc[0])

print("\nData type:")
print(type(medications_df["Medication"].iloc[0]))

List-based recommendation columns parsed successfully.

Example medication:
['SSRIs (e.g., Sertraline, Fluoxetine)', 'Benzodiazepines (e.g., Clonazepam, Alprazolam)', 'SNRIs (e.g., Venlafaxine)', 'Beta-blockers', 'Cognitive Behavioral Therapy (CBT)']

Data type:
<class 'list'>


## Health Recommendation Retrieval

The following function retrieves disease-specific information from the
five recommendation datasets. This forms the grounded knowledge layer
that will later be supplied to the Generative AI component.

In [11]:
def get_health_recommendations(disease):

    disease = normalize_disease_name(disease)

    # Retrieve matching records
    description_row = description_df[
        description_df["Disease"] == disease
    ]

    precautions_row = precautions_df[
        precautions_df["Disease"] == disease
    ]

    medications_row = medications_df[
        medications_df["Disease"] == disease
    ]

    diets_row = diets_df[
        diets_df["Disease"] == disease
    ]

    workout_row = workout_df[
        workout_df["Disease"] == disease
    ]

    # Safety check
    if (description_row.empty or 
        precautions_row.empty or 
        medications_row.empty or 
        diets_row.empty or 
        workout_row.empty):
        return None

    # Extract description
    description = description_row.iloc[0]["Description"]

    # Extract precautions
    precaution_columns = [
        "Precaution_1",
        "Precaution_2",
        "Precaution_3",
        "Precaution_4"
    ]

    precautions = [
        precautions_row.iloc[0][column]
        for column in precaution_columns
        if pd.notna(precautions_row.iloc[0][column])
    ]

    # Extract list-based recommendations
    treatments = medications_row.iloc[0]["Medication"]
    diet = diets_row.iloc[0]["Diet"]
    workouts = workout_row.iloc[0]["Workouts"]

    return {
        "disease": disease,
        "description": description,
        "precautions": precautions,
        "treatments": treatments,
        "diet": diet,
        "workouts": workouts
    }

In [12]:
test_disease = "panic disorder"

recommendations = get_health_recommendations(test_disease)

print("Disease:")
print(recommendations["disease"])

print("\nDescription:")
print(recommendations["description"])

print("\nPrecautions:")
for item in recommendations["precautions"]:
    print("-", item)

print("\nTreatment Information:")
for item in recommendations["treatments"]:
    print("-", item)

print("\nDiet:")
for item in recommendations["diet"]:
    print("-", item)

print("\nWorkout:")
for item in recommendations["workouts"]:
    print("-", item)

Disease:
panic disorder

Description:
Panic disorder is a mental health condition marked by sudden, unexpected panic attacks—intense periods of fear or discomfort—often accompanied by physical symptoms like chest pain, rapid heartbeat, shortness of breath, or dizziness.

Precautions:
- Practice deep breathing
- Avoid caffeine
- Follow therapy plan
- Seek support from loved ones

Treatment Information:
- SSRIs (e.g., Sertraline, Fluoxetine)
- Benzodiazepines (e.g., Clonazepam, Alprazolam)
- SNRIs (e.g., Venlafaxine)
- Beta-blockers
- Cognitive Behavioral Therapy (CBT)

Diet:
- Magnesium-rich foods (spinach, pumpkin seeds, almonds)
- Omega-3 fatty acids (salmon, flaxseeds, walnuts)
- Complex carbs (oats, quinoa)
- Green tea (L-theanine)
- Limit caffeine and sugar

Workout:
- Deep breathing exercises: Calm your mind by focusing on slow, deep breaths
- Yoga: Combines breathing and movement for relaxation
- Mindfulness meditation: Helps reduce anxiety by staying present
- Regular aerobic ex

In [13]:
validation_results = []

for disease in sorted(model_diseases):

    try:
        recommendations = get_health_recommendations(disease)

        is_valid = (
            recommendations is not None
            and bool(recommendations["description"])
            and len(recommendations["precautions"]) > 0
            and len(recommendations["treatments"]) > 0
            and len(recommendations["diet"]) > 0
            and len(recommendations["workouts"]) > 0
        )

        validation_results.append({
            "Disease": disease,
            "Valid": is_valid
        })

    except Exception:
        validation_results.append({
            "Disease": disease,
            "Valid": False
        })


validation_df = pd.DataFrame(validation_results)

valid_count = validation_df["Valid"].sum()
failed_count = len(validation_df) - valid_count

print("Total diseases tested :", len(validation_df))
print("Valid recommendations :", valid_count)
print("Failed recommendations:", failed_count)

if failed_count > 0:
    print("\nDiseases requiring attention:")
    display(validation_df[validation_df["Valid"] == False])

Total diseases tested : 100
Valid recommendations : 100
Failed recommendations: 0


In [14]:
def build_health_guidance_prompt(recommendations):

    disease = recommendations["disease"]
    description = recommendations["description"]
    precautions = recommendations["precautions"]
    treatments = recommendations["treatments"]
    diet = recommendations["diet"]
    workouts = recommendations["workouts"]

    prompt = f"""
You are a health-information assistant.

The disease prediction model has identified the following possible condition:

Disease: {disease}

Use ONLY the grounded information provided below to prepare the response.

DESCRIPTION:
{description}

PRECAUTIONS:
{precautions}

TREATMENT INFORMATION:
{treatments}

DIET:
{diet}

ACTIVITY / WORKOUT:
{workouts}

Instructions:
1. Present the information in clear, simple language.
2. Organize the response into:
   - About the Condition
   - Recommended Precautions
   - Diet Guidance
   - Activity Guidance
   - Treatment Information
3. Do not diagnose the user.
4. Do not prescribe medications or provide dosages.
5. Do not add medications, treatments, or medical facts that are not
   contained in the grounded information.
6. Clearly state that medication decisions should be made with a
   qualified healthcare professional.
7. End with a short statement that the prediction and recommendations
   are informational and are not a substitute for professional
   medical diagnosis or treatment.
"""

    return prompt.strip()

In [15]:
test_recommendations = get_health_recommendations(
    "panic disorder"
)

test_prompt = build_health_guidance_prompt(
    test_recommendations
)

print(test_prompt)

You are a health-information assistant.

The disease prediction model has identified the following possible condition:

Disease: panic disorder

Use ONLY the grounded information provided below to prepare the response.

DESCRIPTION:
Panic disorder is a mental health condition marked by sudden, unexpected panic attacks—intense periods of fear or discomfort—often accompanied by physical symptoms like chest pain, rapid heartbeat, shortness of breath, or dizziness.

PRECAUTIONS:
['Practice deep breathing', 'Avoid caffeine', 'Follow therapy plan', 'Seek support from loved ones']

TREATMENT INFORMATION:
['SSRIs (e.g., Sertraline, Fluoxetine)', 'Benzodiazepines (e.g., Clonazepam, Alprazolam)', 'SNRIs (e.g., Venlafaxine)', 'Beta-blockers', 'Cognitive Behavioral Therapy (CBT)']

DIET:
['Magnesium-rich foods (spinach, pumpkin seeds, almonds)', 'Omega-3 fatty acids (salmon, flaxseeds, walnuts)', 'Complex carbs (oats, quinoa)', 'Green tea (L-theanine)', 'Limit caffeine and sugar']

ACTIVITY / WORK

In [16]:
import os

from dotenv import load_dotenv
from google import genai

In [17]:
load_dotenv(PROJECT_ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if GEMINI_API_KEY:
    print("Gemini API key loaded successfully.")
else:
    print("Gemini API key not found.")

Gemini API key loaded successfully.


In [18]:
client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [19]:
response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=test_prompt
)

generated_guidance = response.text

print(generated_guidance)

### About the Condition
Panic disorder is a mental health condition marked by sudden, unexpected panic attacks—intense periods of fear or discomfort. These attacks are often accompanied by physical symptoms like chest pain, rapid heartbeat, shortness of breath, or dizziness.

### Recommended Precautions
* Practice deep breathing
* Avoid caffeine
* Follow therapy plan
* Seek support from loved ones

### Diet Guidance
* **Magnesium-rich foods:** Spinach, pumpkin seeds, almonds
* **Omega-3 fatty acids:** Salmon, flaxseeds, walnuts
* **Complex carbs:** Oats, quinoa
* **Green tea:** Contains L-theanine
* **General advice:** Limit caffeine and sugar

### Activity Guidance
* **Deep breathing exercises:** Calm your mind by focusing on slow, deep breaths.
* **Yoga:** Combines breathing and movement for relaxation.
* **Mindfulness meditation:** Helps reduce anxiety by staying present.
* **Regular aerobic exercise:** Boosts mood and reduces stress.

### Treatment Information
The following treatme

## Generative AI-Enhanced Health Guidance

The retrieved disease-specific recommendation data is supplied as grounded
context to a Generative AI model. The model restructures this information
into clear and user-friendly health guidance without independently diagnosing
the user or prescribing medication.

In [20]:
def generate_health_guidance(recommendations):

    if recommendations is None:
        return "Health recommendation information is unavailable."

    prompt = build_health_guidance_prompt(recommendations)

    try:
        response = client.models.generate_content(
            model="gemini-flash-latest",
            contents=prompt
        )

        return response.text

    except Exception as error:
        print(f"Generative AI request failed: {error}")
        return None

In [21]:
test_disease = "allergy"

test_recommendations = get_health_recommendations(test_disease)

ai_guidance = generate_health_guidance(test_recommendations)

if ai_guidance:
    print(ai_guidance)

### About the Condition

An allergy occurs when the immune system overreacts to substances such as pollen, food, or medications. This reaction can lead to symptoms like sneezing, itching, a rash, or anaphylaxis.

### Recommended Precautions

*   Apply calamine to affected areas.
*   Cover the area with a bandage.
*   Use ice compresses to help relieve itching.
*   Avoid known allergens.

### Diet Guidance

*   **Elimination diet:** Avoid foods that trigger allergies.
*   **Omega-3 fatty acids:** Eat foods like salmon and flaxseeds.
*   **Vitamin C-rich foods:** Incorporate options like oranges and bell peppers.
*   **Quercetin-rich foods:** Include items like apples and onions.
*   **Probiotics:** Consume foods such as yogurt and kefir.

### Activity Guidance

*   **Indoor workouts:** Exercise indoors to avoid pollen and other triggers.
*   **Yoga:** Practicing yoga can help calm the body and immune system.
*   **Swimming in clean pools:** Swimming can help clear the airways.
*   **Avo

## Integration with Disease Prediction Model

The trained disease prediction model and its supporting artifacts are loaded
to connect symptom-based prediction with the health recommendation and
Generative AI pipeline.

In [22]:
prediction_model = joblib.load(
    MODEL_DIR / "disease_prediction_model.pkl"
)

label_encoder = joblib.load(
    MODEL_DIR / "label_encoder.pkl"
)

symptom_features = joblib.load(
    MODEL_DIR / "symptom_features.pkl"
)

print("Disease prediction artifacts loaded successfully.")
print("Number of symptom features:", len(symptom_features))
print("Number of disease classes:", len(label_encoder.classes_))

Disease prediction artifacts loaded successfully.
Number of symptom features: 230
Number of disease classes: 100


In [23]:
def create_symptom_vector(selected_symptoms):

    # Normalize user-selected symptom names
    selected_symptoms = {
        symptom.strip().lower()
        for symptom in selected_symptoms
    }

    # Create one patient with all 230 symptoms initially absent
    input_data = pd.DataFrame(
        [np.zeros(len(symptom_features), dtype=int)],
        columns=symptom_features
    )

    # Mark selected symptoms as present
    for symptom in selected_symptoms:
        if symptom in input_data.columns:
            input_data.loc[0, symptom] = 1
        else:
            print(f"Warning: Unknown symptom ignored -> {symptom}")

    return input_data

In [24]:
test_symptoms = [
    "anxiety and nervousness",
    "shortness of breath",
    "depressive or psychotic symptoms",
    "chest tightness",
    "palpitations",
    "irregular heartbeat",
    "breathing fast"
]

test_input = create_symptom_vector(test_symptoms)

print("Input shape:", test_input.shape)
print("Number of selected symptoms:", int(test_input.sum(axis=1).iloc[0]))

print("\nActive symptoms:")
print(
    test_input.columns[
        test_input.iloc[0] == 1
    ].tolist()
)

Input shape: (1, 230)
Number of selected symptoms: 7

Active symptoms:
['anxiety and nervousness', 'shortness of breath', 'depressive or psychotic symptoms', 'chest tightness', 'palpitations', 'irregular heartbeat', 'breathing fast']


In [25]:
def predict_disease(selected_symptoms):

    # Convert symptoms into the 230-feature vector
    input_data = create_symptom_vector(selected_symptoms)

    # Predict encoded disease class
    predicted_class = prediction_model.predict(input_data)[0]

    # Convert encoded class back to disease name
    predicted_disease = label_encoder.inverse_transform(
        [predicted_class]
    )[0]

    # Get prediction probabilities
    probabilities = prediction_model.predict_proba(input_data)[0]

    # Confidence of predicted class
    confidence = probabilities[predicted_class]

    return {
        "disease": predicted_disease,
        "confidence": confidence
    }

In [26]:
prediction = predict_disease(test_symptoms)

print("Predicted Disease :", prediction["disease"])
print(
    "Model Confidence  :",
    f"{prediction['confidence'] * 100:.2f}%"
)

Predicted Disease : panic disorder
Model Confidence  : 99.95%


## End-to-End Disease Prediction and Health Recommendation Pipeline

This pipeline integrates the trained machine learning model, the grounded
health recommendation engine, and the Generative AI layer.

Selected symptoms are converted into the model's feature representation,
used to predict a possible disease, and then mapped to disease-specific
health information. The grounded information is subsequently transformed
into structured, user-friendly guidance using Generative AI.

In [27]:
def predict_and_recommend(selected_symptoms):

    # Step 1: Predict disease
    prediction = predict_disease(selected_symptoms)

    disease = prediction["disease"]
    confidence = prediction["confidence"]

    # Step 2: Retrieve grounded health information
    recommendations = get_health_recommendations(disease)

    if recommendations is None:
        return {
            "predicted_disease": disease,
            "confidence": confidence,
            "recommendations": None,
            "ai_guidance": None
        }

    # Step 3: Generate AI-enhanced health guidance
    ai_guidance = generate_health_guidance(recommendations)

    return {
        "predicted_disease": disease,
        "confidence": confidence,
        "recommendations": recommendations,
        "ai_guidance": ai_guidance
    }

In [28]:
test_symptoms = [
    "anxiety and nervousness",
    "shortness of breath",
    "depressive or psychotic symptoms",
    "chest tightness",
    "palpitations",
    "irregular heartbeat",
    "breathing fast"
]

result = predict_and_recommend(test_symptoms)

print("=" * 60)
print("DISEASE PREDICTION")
print("=" * 60)

print("Predicted Disease :", result["predicted_disease"])
print(
    "Model Confidence  :",
    f"{result['confidence'] * 100:.2f}%"
)

print("\n" + "=" * 60)
print("AI-ENHANCED HEALTH GUIDANCE")
print("=" * 60 + "\n")

print(result["ai_guidance"])

DISEASE PREDICTION
Predicted Disease : panic disorder
Model Confidence  : 99.95%

AI-ENHANCED HEALTH GUIDANCE

### About the Condition

Panic disorder is a mental health condition marked by sudden, unexpected panic attacks—intense periods of fear or discomfort. These attacks are often accompanied by physical symptoms such as chest pain, rapid heartbeat, shortness of breath, or dizziness.

### Recommended Precautions

* Practice deep breathing
* Avoid caffeine
* Follow therapy plan
* Seek support from loved ones

### Diet Guidance

* **Magnesium-rich foods:** Spinach, pumpkin seeds, almonds
* **Omega-3 fatty acids:** Salmon, flaxseeds, walnuts
* **Complex carbs:** Oats, quinoa
* **Green tea:** Contains L-theanine
* Limit caffeine and sugar intake

### Activity Guidance

* **Deep breathing exercises:** Calm your mind by focusing on slow, deep breaths.
* **Yoga:** Combines breathing and movement for relaxation.
* **Mindfulness meditation:** Helps reduce anxiety by staying present.
* **Reg